In [1]:
# ============================================================
# EXPERIMENT 3
# EcoBotX-Light + CBAM
# ============================================================

from pathlib import Path
import torch
import yaml

from ultralytics import YOLO

print("=" * 70)
print("EXPERIMENT 3 — EcoBotX-Light + CBAM")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

EXPERIMENT 3 — EcoBotX-Light + CBAM
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# ============================================================
# LOCATE CBAM IN ULTRALYTICS 8.4.126
# ============================================================

import ultralytics
import ultralytics.nn.modules as modules
import ultralytics.nn.tasks as tasks
import inspect

print("Ultralytics version:", ultralytics.__version__)

print("\nSearching for CBAM...")

# Check modules package
print("\nCBAM in ultralytics.nn.modules:")
print(hasattr(modules, "CBAM"))

# Check tasks namespace
print("\nCBAM in ultralytics.nn.tasks:")
print(hasattr(tasks, "CBAM"))

# Search loaded module attributes
print("\nPossible CBAM locations:")

for module_name, module in [
    ("ultralytics.nn.modules", modules),
    ("ultralytics.nn.tasks", tasks),
]:

    for name in dir(module):
        if "CBAM" in name.upper():
            print(
                f"{module_name}.{name}",
                "->",
                getattr(module, name)
            )

Ultralytics version: 8.4.126

Searching for CBAM...

CBAM in ultralytics.nn.modules:
True

CBAM in ultralytics.nn.tasks:
False

Possible CBAM locations:
ultralytics.nn.modules.CBAM -> <class 'ultralytics.nn.modules.conv.CBAM'>


In [3]:
# ============================================================
# REGISTER CBAM — ULTRALYTICS 8.4.126
# ============================================================

import ultralytics.nn.modules as modules
import ultralytics.nn.tasks as tasks

# Get CBAM from the location discovered in your installation
CBAM = modules.CBAM

# Register it in the namespace used by parse_model()
tasks.CBAM = CBAM

print("=" * 70)
print("CBAM REGISTRATION")
print("=" * 70)

print("CBAM class      :", CBAM)
print("modules.CBAM    :", hasattr(modules, "CBAM"))
print("tasks.CBAM      :", hasattr(tasks, "CBAM"))

print("\n[OK] CBAM registered with Ultralytics parser.")

CBAM REGISTRATION
CBAM class      : <class 'ultralytics.nn.modules.conv.CBAM'>
modules.CBAM    : True
tasks.CBAM      : True

[OK] CBAM registered with Ultralytics parser.


In [4]:
# ============================================================
# EXPERIMENT 3 — DATASET CONFIGURATION
# ============================================================

from pathlib import Path

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO_FINAL\dataset.yaml"
)

print("=" * 70)
print("DATASET CONFIGURATION")
print("=" * 70)

print("Dataset YAML:", DATASET_YAML)

if not DATASET_YAML.exists():
    raise FileNotFoundError(
        f"Dataset YAML not found:\n{DATASET_YAML}"
    )

print("[OK] Dataset YAML found.")

DATASET CONFIGURATION
Dataset YAML: G:\EcoBotX_YOLO_FINAL\dataset.yaml
[OK] Dataset YAML found.


In [5]:
# ============================================================
# INSPECT CBAM IMPLEMENTATION
# ============================================================

import inspect
from ultralytics.nn.modules import CBAM

print("=" * 70)
print("CBAM IMPLEMENTATION")
print("=" * 70)

print("CBAM class:")
print(CBAM)

print("\nCBAM constructor:")
print(inspect.signature(CBAM))

print("\nCBAM source:")
print(inspect.getsource(CBAM))

CBAM IMPLEMENTATION
CBAM class:
<class 'ultralytics.nn.modules.conv.CBAM'>

CBAM constructor:
(c1, kernel_size=7)

CBAM source:
class CBAM(nn.Module):
    """Convolutional Block Attention Module.

    Combines channel and spatial attention mechanisms for comprehensive feature refinement.

    Attributes:
        channel_attention (ChannelAttention): Channel attention module.
        spatial_attention (SpatialAttention): Spatial attention module.
    """

    def __init__(self, c1, kernel_size=7):
        """Initialize CBAM with given parameters.

        Args:
            c1 (int): Number of input channels.
            kernel_size (int): Size of the convolutional kernel for spatial attention.
        """
        super().__init__()
        self.channel_attention = ChannelAttention(c1)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        """Apply channel and spatial attention sequentially to input tensor.

        Args:
            x (torch.Te

In [6]:
# ============================================================
# EXPERIMENT 3 — ECOBOTX-LIGHT + CBAM
# CORRECTED MODEL CONFIGURATION
# ============================================================

import yaml
from pathlib import Path

MODEL_YAML = Path(
    r"G:\EcoBotX_YOLO_training\experiment3_ecobotx_cbam.yaml"
)

model_config = {

    "nc": 4,

    "depth_multiple": 0.33,
    "width_multiple": 0.25,

    # ========================================================
    # BACKBONE
    # ========================================================

    "backbone": [

        [-1, 1, "Conv", [64, 3, 2]],

        [-1, 1, "Conv", [128, 3, 2]],

        [-1, 3, "C2f", [128, True]],

        [-1, 1, "Conv", [256, 3, 2]],

        [-1, 6, "C2f", [256, True]],

        # Actual channels after width scaling:
        # 256 * 0.25 = 64
        [-1, 1, "CBAM", [64, 7]],

        [-1, 1, "Conv", [512, 3, 2]],

        [-1, 6, "C2f", [512, True]],

        # Actual channels:
        # 512 * 0.25 = 128
        [-1, 1, "CBAM", [128, 7]],

        [-1, 1, "Conv", [1024, 3, 2]],

        [-1, 3, "C2f", [1024, True]],

        [-1, 1, "SPPF", [1024, 5]],
    ],

    # ========================================================
    # HEAD
    # ========================================================

    "head": [

        # P5 -> P4
        [-1, 1, "nn.Upsample", ["None", 2, "nearest"]],

        [[-1, 7], 1, "Concat", [1]],

        [-1, 3, "C2f", [512]],

        # P4 channels = 128
        [-1, 1, "CBAM", [128, 7]],

        # P4 -> P3
        [-1, 1, "nn.Upsample", ["None", 2, "nearest"]],

        [[-1, 4], 1, "Concat", [1]],

        [-1, 3, "C2f", [256]],

        # P3 channels = 64
        [-1, 1, "CBAM", [64, 7]],

        # P3 -> P4
        [-1, 1, "Conv", [256, 3, 2]],

        [[-1, 15], 1, "Concat", [1]],

        [-1, 3, "C2f", [512]],

        # P4 -> P5
        [-1, 1, "Conv", [512, 3, 2]],

        [[-1, 11], 1, "Concat", [1]],

        [-1, 3, "C2f", [1024]],

        # Detection
        [[19, 22, 25], 1, "Detect", ["nc"]],
    ],

    "ch": 3,
}

# Save YAML
with open(MODEL_YAML, "w") as f:
    yaml.dump(
        model_config,
        f,
        sort_keys=False
    )

print("=" * 70)
print("CORRECTED CBAM YAML CREATED")
print("=" * 70)

print("Path:")
print(MODEL_YAML)

print("\nCBAM channel configuration:")
print("Backbone CBAM-1 : 64")
print("Backbone CBAM-2 : 128")
print("Head CBAM-1     : 128")
print("Head CBAM-2     : 64")

CORRECTED CBAM YAML CREATED
Path:
G:\EcoBotX_YOLO_training\experiment3_ecobotx_cbam.yaml

CBAM channel configuration:
Backbone CBAM-1 : 64
Backbone CBAM-2 : 128
Head CBAM-1     : 128
Head CBAM-2     : 64


In [7]:
# ============================================================
# BUILD EXPERIMENT 3
# ============================================================

print("=" * 70)
print("BUILDING EXPERIMENT 3")
print("EcoBotX-Light + CBAM")
print("=" * 70)

custom_cbam_model = YOLO(
    str(MODEL_YAML)
)

print()
print("[OK] Model created successfully.")

print()
print("=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

custom_cbam_model.info(verbose=True)

BUILDING EXPERIMENT 3
EcoBotX-Light + CBAM

[OK] Model created successfully.

MODEL INFORMATION
experiment3_ecobotx_cbam summary: 150 layers, 3,053,364 parameters, 3,053,348 gradients, 8.2 GFLOPs


(150, 3053364, 3053348, 8.200594688)

In [8]:
# ============================================================
# EXPERIMENT 3 — CBAM LAYER VERIFICATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 3 — CBAM LAYERS")
print("=" * 70)

cbam_count = 0

for i, layer in enumerate(custom_cbam_model.model.model):

    if layer.__class__.__name__ == "CBAM":

        cbam_count += 1

        print(f"\nCBAM #{cbam_count}")
        print(f"Layer index : {i}")
        print(f"Module      : {layer}")

print("\n" + "=" * 70)
print(f"TOTAL CBAM LAYERS: {cbam_count}")
print("=" * 70)

if cbam_count == 4:
    print("[OK] All 4 CBAM modules detected.")
else:
    print("[WARNING] Expected 4 CBAM modules.")

EXPERIMENT 3 — CBAM LAYERS

CBAM #1
Layer index : 5
Module      : CBAM(
  (channel_attention): ChannelAttention(
    (pool): AdaptiveAvgPool2d(output_size=1)
    (fc): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (act): Sigmoid()
  )
  (spatial_attention): SpatialAttention(
    (cv1): Conv2d(2, 1, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
    (act): Sigmoid()
  )
)

CBAM #2
Layer index : 8
Module      : CBAM(
  (channel_attention): ChannelAttention(
    (pool): AdaptiveAvgPool2d(output_size=1)
    (fc): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1))
    (act): Sigmoid()
  )
  (spatial_attention): SpatialAttention(
    (cv1): Conv2d(2, 1, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
    (act): Sigmoid()
  )
)

CBAM #3
Layer index : 15
Module      : CBAM(
  (channel_attention): ChannelAttention(
    (pool): AdaptiveAvgPool2d(output_size=1)
    (fc): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1))
    (act): Sigmoid()
  )
  (spatial

In [9]:
# ============================================================
# EXPERIMENT 3 — DUMMY FORWARD PASS
# ============================================================

import torch

print("=" * 70)
print("TESTING FORWARD PASS")
print("=" * 70)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

custom_cbam_model.model.to(device)

dummy = torch.randn(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = custom_cbam_model.model(dummy)

print("[OK] Forward pass completed successfully.")
print("Device:", device)

TESTING FORWARD PASS
[OK] Forward pass completed successfully.
Device: cuda:0


In [10]:
# ============================================================
# EXPERIMENT 3 — TRAINING
# EcoBotX-Light + CBAM
# ============================================================

PROJECT_DIR = r"G:\EcoBotX_YOLO_training"

print("=" * 70)
print("EXPERIMENT 3 TRAINING")
print("EcoBotX-Light + CBAM")
print("=" * 70)

results_exp3 = custom_cbam_model.train(

    # Dataset
    data=str(DATASET_YAML),

    # Training
    epochs=100,
    imgsz=640,
    batch=8,

    # GPU
    device=0,

    # Data loading
    workers=4,

    # Train from scratch
    pretrained=False,

    # Optimizer
    optimizer="auto",

    # Early stopping
    patience=20,

    # Saving
    save=True,
    save_period=10,

    # Plots
    plots=True,

    # Output
    project=PROJECT_DIR,
    name="experiment3_ecobotx_cbam",

    verbose=True
)

EXPERIMENT 3 TRAINING
EcoBotX-Light + CBAM
New https://pypi.org/project/ultralytics/8.4.127 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\EcoBotX_YOLO_FINAL\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=Non

In [11]:
# ============================================================
# EXPERIMENT 3 — EcoBotX-Light + CBAM
# TEST SET EVALUATION
# ============================================================

from ultralytics import YOLO
from pathlib import Path

print("=" * 70)
print("EXPERIMENT 3 — EcoBotX-Light + CBAM TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment3_ecobotx_cbam\weights\best.pt"
)

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO_FINAL\dataset.yaml"
)

PROJECT_DIR = Path(
    r"G:\EcoBotX_YOLO_training"
)

# ------------------------------------------------------------
# Load trained model
# ------------------------------------------------------------

model_exp3 = YOLO(str(MODEL_PATH))

print("\nModel loaded successfully.")
print("Model:", MODEL_PATH)

# ------------------------------------------------------------
# Model information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

model_exp3.info(verbose=False)

# ------------------------------------------------------------
# TEST EVALUATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING TEST EVALUATION")
print("=" * 70)

test_results_exp3 = model_exp3.val(
    data=str(DATASET_YAML),

    split="test",

    imgsz=640,

    batch=8,

    device=0,

    workers=4,

    plots=True,

    save_json=True,

    project=str(PROJECT_DIR),

    name="experiment3_ecobotx_cbam_test",

    verbose=True
)

# ------------------------------------------------------------
# Extract metrics
# ------------------------------------------------------------

precision = float(test_results_exp3.box.mp)
recall = float(test_results_exp3.box.mr)
map50 = float(test_results_exp3.box.map50)
map5095 = float(test_results_exp3.box.map)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("EXPERIMENT 3 — CBAM TEST RESULTS")
print("=" * 70)

print(f"Parameters : 3,048,164")
print(f"GFLOPs     : 8.1")

print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"mAP50      : {map50:.4f}")
print(f"mAP50-95   : {map5095:.4f}")

print("=" * 70)

EXPERIMENT 3 — EcoBotX-Light + CBAM TEST EVALUATION

Model loaded successfully.
Model: G:\EcoBotX_YOLO_training\experiment3_ecobotx_cbam\weights\best.pt

MODEL INFORMATION

RUNNING TEST EVALUATION
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
experiment3_ecobotx_cbam summary (fused): 93 layers, 3,048,164 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.71.1 ms, read: 20.97.1 MB/s, size: 7.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning G:\EcoBotX_YOLO_FINAL\labels\test.cache... 1115 images, 216 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1115/1115  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 11.6it/s 12.0s0.2s
                   all       1115        899      0.909      0.929      0.961      0.632
            